# Official-style CIFAR-10 unlearning score

This notebook compares each unlearning method against retain-only models trained from scratch. It reports official forgetting quality F, retain/test utility ratios and final score, plus runtime and the simple loss-based MIA diagnostic. The checkpoint's exact training duration and schedule are undocumented; the selected scratch recipe is recorded in `TrainingRecipe` and is a local reference, not an exact reproduction.

Start with N=8 and a fixed forget-set subsample for development. Keep N, seeds, and the forget sample unchanged for every method. Use the full forget set for final reporting.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "unlearn").is_dir():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")
sys.path.insert(0, str(repo_root / "src"))

from unlearn.data import get_cifar10_loaders
from unlearn.device import DEVICE
from unlearn.evaluate import TrainingRecipe, run_method, score, train_or_load_retrained
from unlearn.models import load_cifar10_model
from unlearn.methods import Kookmin_2, finetune

print("Device:", DEVICE)

In [ ]:
loaders = get_cifar10_loaders(
    root=str(repo_root / "Notebooks" / "data"),
    forget_idx_path=str(repo_root / "Notebooks" / "forget_idx.npy"),
    batch_size=128,
    num_workers=2,
    seed=42,
)
retain_loader = loaders["retain"]
forget_loader = loaders["forget"]
test_loader = loaders["test"]
val_loader = loaders["val"]
original_model = load_cifar10_model(repo_root / "weights_resnet18_cifar10.pth")

print("Forget-set size:", len(forget_loader.dataset))
print("Retain-set size:", len(retain_loader.dataset))
print("Test-set size:", len(test_loader.dataset))

In [ ]:
N = 8
SEEDS = tuple(range(100, 100 + N))
FORGET_SAMPLE_SIZE = 256  # Set to None for final full-forget-set numbers.
FORGET_SAMPLE_SEED = 31415

recipe = TrainingRecipe(
    epochs=30,  # Chosen: the checkpoint's epoch count was not documented.
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    batch_size=128,
    num_workers=2,
    scheduler="cosine",
    augment=True,
)

retrained = train_or_load_retrained(
    N,
    recipe,
    retain_loader=retain_loader,
    forget_loader=forget_loader,
    test_loader=test_loader,
    seeds=SEEDS,
    cache_path=repo_root / "outputs" / "retrain_confs.npz",
    max_forget_examples=FORGET_SAMPLE_SIZE,
    forget_sample_seed=FORGET_SAMPLE_SEED,
)
print(f"Reference cache ready: N={N}, forget examples={retrained.forget_confs.shape[1]}, sample seed={FORGET_SAMPLE_SEED}")

In [ ]:
# Add later competition methods here once their adapters are ready.
method_specs = [
    ("Fine-tune", finetune.unlearning),
    ("Kookmin_2", Kookmin_2.unlearning),
]

rows = []
for method_name, method_fn in method_specs:
    result = run_method(
        method_fn,
        N,
        SEEDS,
        original_model=original_model,
        retain_loader=retain_loader,
        forget_loader=forget_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        max_forget_examples=FORGET_SAMPLE_SIZE,
        forget_sample_seed=FORGET_SAMPLE_SEED,
    )
    row = score(result, retrained)
    row["method"] = method_name
    rows.append(row)

columns = ["method", "F", "retain_acc", "test_acc", "final_score", "runtime_sec", "runtime_ratio", "loss_mia_acc", "N"]
print(" | ".join(columns))
print(" | ".join(["---"] * len(columns)))
for row in rows:
    print(" | ".join(
        f"{row[column]:.5f}" if isinstance(row[column], float) else str(row[column])
        for column in columns
    ))